# Proyecto P08 – Clasificación de alertas de seguridad del SIEM

**Estudiante:** _(nombre completo)_

Notebook base obligatorio. Se pueden agregar celdas dentro de cada sección, pero no se debe cambiar el orden ni los títulos de las secciones. El protocolo de evaluación (sección 8 del enunciado) es obligatorio.

## 1. Carga de datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42
crudo = pd.read_csv("data/alertas_siem.csv")
print(crudo.shape)
crudo.head()

## 2. Comprensión de los datos

Describir tipos de datos, valores faltantes, registros repetidos, valores imposibles, categorías y registros que no correspondan a casos reales. Cada observación debe estar respaldada por una salida de código.

## 3. Preparación de los datos

Registrar cada decisión con el formato:

- **Decisión:** qué se hizo.
- **Evidencia:** qué salida de código respalda la decisión.
- **Justificación:** por qué se eligió esa opción y no otra.

El resultado de esta sección debe ser el DataFrame final `df`, la matriz `X` y la variable objetivo `y`.

## 4. Contraste de hipótesis

Responder H1 a H4 indicando si **se cumple** o **no se cumple**, con evidencia.

## 5. Modelado

Aplicar el protocolo obligatorio:

- `train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)`
- Preprocesamiento dentro de `Pipeline` / `ColumnTransformer`.
- `LogisticRegression(max_iter=1000, class_weight="balanced")`
- `RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)`

## 6. Evaluación

Incluir: exactitud, precisión y recall de ambos modelos con umbral 0.5; matrices de confusión; análisis de los umbrales 0.1 a 0.9 del modelo recomendado; validación cruzada con `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` y métrica recall.

## 7. Conclusiones

Modelo recomendado, justificación con las condiciones del caso y limitaciones.

## 8. Ficha de resultados

Completar cada campo **con código** (variables calculadas en las secciones anteriores), no con valores escritos a mano. Ejecutar la celda para generar `ficha_resultados.json`, que se entrega junto con el notebook.

In [ ]:
FICHA = {
    "proyecto": "P08",
    "estudiante": "",                        # nombre completo
    "registros_originales": None,                     # registros del archivo original
    "duplicados_eliminados": None,                    # registros duplicados eliminados
    "registros_no_reales_excluidos": None,            # registros excluidos por no corresponder a casos reales
    "registros_finales": None,                        # registros del conjunto final usado para modelar
    "positivos_finales": None,                        # alertas que corresponden a incidentes reales (objetivo = 1) en el conjunto final
    "valores_imposibles_corregidos": None,            # total de valores imposibles detectados y corregidos/excluidos
    "alertas_severidad_alta": None,                   # alertas del conjunto final con severidad alta (3)
    "tasa_incidente_severidad_alta": {},              # tasa de incidentes reales en severidad alta (0 a 1)
    "tasa_incidente_lista_negra": {},                 # tasa de incidentes con IP en lista negra (0 a 1)
    "tasa_incidente_sin_lista_negra": {},             # tasa de incidentes sin IP en lista negra (0 a 1)
    "tasa_incidente_por_regla": {},                   # {regla: tasa de incidentes}
    "columnas_excluidas": [],                         # columnas ORIGINALES que no se usan como entrada del modelo
    "variables_modelo": [],                           # nombres de las columnas de X
    "H1_se_cumple": None,                             # True / False
    "H2_se_cumple": None,                             # True / False
    "H3_se_cumple": None,                             # True / False
    "H4_se_cumple": None,                             # True / False
    "positivos_en_prueba": None,                      # positivos en y_test
    "logistica_umbral_05": {"exactitud": None, "precision": None, "recall": None},  # exactitud, precision y recall con umbral 0.5
    "random_forest_umbral_05": {"exactitud": None, "precision": None, "recall": None},  # exactitud, precision y recall con umbral 0.5
    "modelo_recomendado": "",                         # "Regresión logística", "Random Forest" u otro
    "umbral_recomendado": None,                       # umbral de decisión recomendado
    "precision_umbral_recomendado": None,             # precisión con el umbral recomendado
    "recall_umbral_recomendado": None,                # recall con el umbral recomendado
    "recall_cv_media_modelo_recomendado": None,       # recall medio de la validación cruzada (5 folds)
}

In [ ]:
import json

def _a_python(v):
    if isinstance(v, dict):
        return {k: _a_python(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, set, pd.Index)):
        return [_a_python(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return round(float(v), 4)
    if isinstance(v, np.bool_):
        return bool(v)
    return v

def _vacios(d, prefijo=""):
    for k, v in d.items():
        if isinstance(v, dict) and v:
            yield from _vacios(v, f"{prefijo}{k}.")
        elif v is None or v == "" or v == [] or v == {}:
            yield prefijo + k

pendientes = list(_vacios(FICHA))
print("Campos pendientes:", pendientes if pendientes else "ninguno")
with open("ficha_resultados.json", "w", encoding="utf-8") as f:
    json.dump(_a_python(FICHA), f, ensure_ascii=False, indent=2)
print("Ficha guardada en ficha_resultados.json")